In [37]:
# object_detection_tutorial.ipynb

In [1]:
!pip install -U --pre tensorflow=="2.*"

In [13]:
!pip install tf_slim
!pip install pycocotools

In [3]:
!cd models/research/ && protoc object_detection/protos/*.proto --python_out=.

In [4]:
!cd models/research/ && pip install .

Processing /Volumes/Dylan/Lego_Coco_training/models/research
  Preparing metadata (setup.py) ... done
  Created wheel for object_detection: filename=object_detection-0.1-py3-none-any.whl size=1655479 sha256=ca77f99d86c0d5d0e972b9f48542f7583105653866d61a3af744ac4c9b1e4781
  Stored in directory: /private/var/folders/9k/5h0j9gh93pb9ldsyknc0c9h80000gn/T/pip-ephem-wheel-cache-tqa2te6e/wheels/f4/4f/f5/688d7c185aee96c936ad8326e3421af98355ee7c2d7a4965f2
Successfully built object_detection
  Attempting uninstall: object_detection
    Found existing installation: object_detection 0.1
    Uninstalling object_detection-0.1:
      Successfully uninstalled object_detection-0.1


In [14]:
import numpy as np
import pathlib
import os
import six.moves.urllib as urllib
import sys
import tarfile
import tensorflow as tf
import zipfile

from collections import defaultdict
from io import StringIO
from matplotlib import pyplot as plt
from PIL import Image
from IPython.display import display

In [15]:
from object_detection.utils import ops as utils_ops
from object_detection.utils import label_map_util
from object_detection.utils import visualization_utils as vis_util

In [16]:
utils_ops.tf = tf.compat.v1

In [17]:
tf.gfile = tf.io.gfile

In [31]:
def load_model(model_name):
    base_url = 'http://download.tensorflow.org/models/object_detection/'
    model_file = model_name + '.tar.gz'
    model_dir = tf.keras.utils.get_file(
        fname=model_name,
        origin=base_url + model_file,
        untar=True,
        cache_dir="/Volumes/Dylan/Lego_Coco_training")

    model_dir = pathlib.Path(model_dir)/"ssd_mobilenet_v1_coco_2017_11_17/saved_model"
    print("MODEL DIR: ", model_dir)
    model = tf.saved_model.load(str(model_dir))

    return model

In [24]:
!pwd

/Volumes/Dylan/Lego_Coco_training


In [19]:
PATH_TO_LABELS = 'label_map_legofriend.pbtxt'
category_index = label_map_util.create_category_index_from_labelmap(PATH_TO_LABELS, use_display_name=True)

In [27]:
PATH_TO_TEST_IMAGES_DIR = pathlib.Path('Simple_Test_photos/')
TEST_IMAGE_PATHS = sorted(list(PATH_TO_TEST_IMAGES_DIR.glob("*.jpg")))
TEST_IMAGE_PATHS

[PosixPath('Simple_Test_photos/lego-00.11.20-09.18.2024 copy.jpg'),
 PosixPath('Simple_Test_photos/lego-07.08.20-09.18.2024 copy.jpg')]

In [32]:
model_name = 'ssd_mobilenet_v1_coco_2017_11_17'
detection_model = load_model(model_name)

MODEL DIR:  /Volumes/Dylan/Lego_Coco_training/datasets/ssd_mobilenet_v1_coco_2017_11_17/ssd_mobilenet_v1_coco_2017_11_17/saved_model
INFO:tensorflow:Saver not created because there are no variables in the graph to restore


In [33]:
print(detection_model.signatures['serving_default'].inputs)

[<tf.Tensor 'image_tensor:0' shape=(None, None, None, 3) dtype=uint8>]


In [34]:
detection_model.signatures['serving_default'].output_dtypes

{'num_detections': tf.float32,
 'detection_boxes': tf.float32,
 'detection_classes': tf.float32,
 'detection_scores': tf.float32}

In [35]:
detection_model.signatures['serving_default'].output_shapes

{'num_detections': TensorShape([None]),
 'detection_boxes': TensorShape([None, 100, 4]),
 'detection_classes': TensorShape([None, 100]),
 'detection_scores': TensorShape([None, 100])}

In [36]:
def run_inference_for_single_image(model, image):
  image = np.asarray(image)
  # The input needs to be a tensor, convert it using `tf.convert_to_tensor`.
  input_tensor = tf.convert_to_tensor(image)
  # The model expects a batch of images, so add an axis with `tf.newaxis`.
  input_tensor = input_tensor[tf.newaxis,...]

  # Run inference
  model_fn = model.signatures['serving_default']
  output_dict = model_fn(input_tensor)

  # All outputs are batches tensors.
  # Convert to numpy arrays, and take index [0] to remove the batch dimension.
  # We're only interested in the first num_detections.
  num_detections = int(output_dict.pop('num_detections'))
  output_dict = {key:value[0, :num_detections].numpy() 
                 for key,value in output_dict.items()}
  output_dict['num_detections'] = num_detections

  # detection_classes should be ints.
  output_dict['detection_classes'] = output_dict['detection_classes'].astype(np.int64)
   
  # Handle models with masks:
  if 'detection_masks' in output_dict:
    # Reframe the the bbox mask to the image size.
    detection_masks_reframed = utils_ops.reframe_box_masks_to_image_masks(
              output_dict['detection_masks'], output_dict['detection_boxes'],
               image.shape[0], image.shape[1])      
    detection_masks_reframed = tf.cast(detection_masks_reframed > 0.5,
                                       tf.uint8)
    output_dict['detection_masks_reframed'] = detection_masks_reframed.numpy()
    
  return output_dict

In [38]:
def show_inference(model, image_path):
  # the array based representation of the image will be used later in order to prepare the
  # result image with boxes and labels on it.
  image_np = np.array(Image.open(image_path))
  # Actual detection.
  output_dict = run_inference_for_single_image(model, image_np)
  # Visualization of the results of a detection.
  vis_util.visualize_boxes_and_labels_on_image_array(
      image_np,
      output_dict['detection_boxes'],
      output_dict['detection_classes'],
      output_dict['detection_scores'],
      category_index,
      instance_masks=output_dict.get('detection_masks_reframed', None),
      use_normalized_coordinates=True,
      line_thickness=8)

  display(Image.fromarray(image_np))

In [ ]:
for image_path in TEST_IMAGE_PATHS:
    show_inference